# 03 — Feature Engineering for Forecasting

**Gold Nexus Alpha — professor-style forecasting pipeline**

Purpose of this notebook:

1. Load the weekday-clean matrix created by Notebook 01.
2. Load the cutoff/window governance created by Notebook 02 when available.
3. Create professor-safe gold lag, return, moving-average, and rolling-volatility features.
4. Create the long univariate dataset for baseline/classical time-series models.
5. Create the core multivariate dataset for regression, SARIMAX, and XGBoost.
6. Exclude `high_yield` from the main multivariate dataset.
7. Export CSV/JSON artifacts for later notebooks and the Vercel frontend.
8. Push outputs to GitHub.

Professor-safe rule: engineered gold features use **prior observed gold values only**. Rolling features are shifted by one row before calculation so the current target value is not used to predict itself.


In [ ]:
# ======================================================================================
# CELL 1 — REPO SYNC AND CLEAN RESET
# ======================================================================================
# Purpose:
# - Clone the GitHub repository into Colab.
# - Load GITHUB_TOKEN from Colab Secrets.
# - Keep the same repo flow as earlier notebooks.
# ======================================================================================

import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO_OWNER = "rathee000001"
REPO_NAME  = "nyit-gold-intelligence-2026"
BRANCH     = "main"

BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / REPO_NAME

GITHUB_TOKEN = ""
if userdata is not None:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
else:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

def run_cmd(args, cwd=None, allow_fail=False, mask_token=True):
    """Run a command safely without printing the raw GitHub token."""
    printable = " ".join(str(a) for a in args)
    if mask_token and GITHUB_TOKEN:
        printable = printable.replace(GITHUB_TOKEN, "***GITHUB_TOKEN***")
    print(f">> {printable}")
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {printable}")
    return p

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print(f"🧹 Removed old project folder: {PROJECT_DIR}")

if GITHUB_TOKEN:
    CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    CLONE_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    print("⚠️ GITHUB_TOKEN not found. Clone may work if repo is public, but push will be skipped.")

run_cmd(["git", "clone", "--branch", BRANCH, CLONE_URL, str(PROJECT_DIR)])

run_cmd(["git", "config", "user.email", "colab-gold-nexus@nyit.local"], cwd=str(PROJECT_DIR))
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=str(PROJECT_DIR))

print("✅ Repo ready:", PROJECT_DIR)
print("✅ Branch:", BRANCH)
print("✅ Timestamp UTC:", datetime.now(timezone.utc).isoformat())


In [ ]:
# ======================================================================================
# CELL 2 — DEPENDENCIES
# ======================================================================================
# Purpose:
# - Load standard data engineering libraries.
# - No model fitting happens in this notebook.
# ======================================================================================

import json
import math
import glob
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

print("✅ Dependencies loaded")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


In [ ]:
# ======================================================================================
# CELL 3 — PATH SETUP, INPUT DETECTION, AND LOCKED WINDOW CONFIG
# ======================================================================================
# Purpose:
# - Locate the weekday-clean matrix from Notebook 01.
# - Load governance artifacts from Notebook 02 when present.
# - Define locked dataset windows and split boundaries.
# ======================================================================================

PROJECT_DIR = Path(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
SPLITS_DIR = DATA_DIR / "splits"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
DATA_ARTIFACTS_DIR = ARTIFACTS_DIR / "data"
GOVERNANCE_DIR = ARTIFACTS_DIR / "governance"

for folder in [ALIGNED_DIR, SPLITS_DIR, DATA_ARTIFACTS_DIR, GOVERNANCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def read_json_if_exists(path: Path, default=None):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

forecast_status = read_json_if_exists(GOVERNANCE_DIR / "forecast_status.json", default={})
model_window_plan = read_json_if_exists(GOVERNANCE_DIR / "model_window_plan.json", default={})

OFFICIAL_FORECAST_CUTOFF_DATE = pd.to_datetime(
    forecast_status.get("official_forecast_cutoff_date", "2026-03-31")
)

# Locked windows from the project plan.
LONG_UNIVARIATE_WINDOW = {
    "dataset_id": "long_univariate",
    "start": pd.to_datetime("1968-01-04"),
    "end": OFFICIAL_FORECAST_CUTOFF_DATE,
    "target": "gold_price",
    "purpose": "Naive, Moving Average, Exponential Smoothing, ARIMA, Prophet optional"
}

CORE_MULTIVARIATE_WINDOW = {
    "dataset_id": "core_multivariate",
    "start": pd.to_datetime("2006-01-02"),
    "end": OFFICIAL_FORECAST_CUTOFF_DATE,
    "target": "gold_price",
    "purpose": "Regression, SARIMAX, XGBoost main candidate"
}

SPLIT_RULES = {
    "long_univariate": {
        "train": ("1968-01-04", "2018-12-31"),
        "validation": ("2019-01-01", "2022-12-30"),
        "test": ("2023-01-02", "2026-03-31")
    },
    "core_multivariate": {
        "train": ("2006-01-02", "2018-12-31"),
        "validation": ("2019-01-01", "2022-12-30"),
        "test": ("2023-01-02", "2026-03-31")
    }
}

CORE_FACTORS = [
    "real_yield",
    "nominal_yield",
    "tips_curve",
    "fed_bs",
    "m2_supply",
    "inflation",
    "usd_index",
    "eur_usd",
    "jpy_usd",
    "vix_index",
    "fin_stress",
    "gpr_index",
    "policy_unc",
    "oil_wti",
    "ppi_index",
    "gld_tonnes",
    "unrate",
    "ind_prod",
    "cap_util",
]

ENGINEERED_GOLD_FEATURES = [
    "gold_lag_1",
    "gold_lag_5",
    "gold_lag_20",
    "gold_return_1",
    "gold_return_5",
    "gold_ma_5",
    "gold_ma_20",
    "gold_ma_60",
    "gold_volatility_20",
]

XGBOOST_MAIN_FEATURES = ENGINEERED_GOLD_FEATURES + CORE_FACTORS

# Candidate input paths. Prefer the Notebook 01 weekday-clean output.
candidate_matrix_paths = [
    ALIGNED_DIR / "weekday_clean_matrix.csv",
    DATA_DIR / "aligned" / "weekday_clean_matrix.csv",
    DATA_DIR / "Gold_Matrix_M3_Daily_2026-04-30.csv",
    PROJECT_DIR / "Gold_Matrix_M3_Daily_2026-04-30.csv",
    Path("/content/Gold_Matrix_M3_Daily_2026-04-30.csv"),
    Path("/content/Gold_Matrix_M3_Daily_2026-04-30 (4).csv"),
]

# Also scan common locations for the latest raw matrix if the standard paths are missing.
candidate_matrix_paths += [Path(p) for p in glob.glob("/content/**/Gold_Matrix_M3_Daily_2026-04-30*.csv", recursive=True)]
candidate_matrix_paths += [Path(p) for p in glob.glob(str(PROJECT_DIR / "**/Gold_Matrix_M3_Daily_2026-04-30*.csv"), recursive=True)]

INPUT_MATRIX_PATH = None
for path in candidate_matrix_paths:
    if path.exists():
        INPUT_MATRIX_PATH = path
        break

if INPUT_MATRIX_PATH is None:
    raise FileNotFoundError(
        "Could not find weekday_clean_matrix.csv or Gold_Matrix_M3_Daily_2026-04-30.csv. "
        "Run Notebook 01 first, or upload the current matrix CSV to Colab."
    )

print("✅ Input matrix path:", INPUT_MATRIX_PATH)
print("✅ Official forecast cutoff:", OFFICIAL_FORECAST_CUTOFF_DATE.date())
print("✅ Long univariate window:", LONG_UNIVARIATE_WINDOW["start"].date(), "to", LONG_UNIVARIATE_WINDOW["end"].date())
print("✅ Core multivariate window:", CORE_MULTIVARIATE_WINDOW["start"].date(), "to", CORE_MULTIVARIATE_WINDOW["end"].date())
print("✅ Core factors:", len(CORE_FACTORS))
print("✅ Engineered gold features:", len(ENGINEERED_GOLD_FEATURES))


In [ ]:
# ======================================================================================
# CELL 4 — MAIN FEATURE ENGINEERING LOGIC
# ======================================================================================
# Purpose:
# - Create lagged, shifted, and rolling gold features.
# - Create the model-ready long univariate dataset.
# - Create the model-ready core multivariate dataset.
# - Preserve leakage-control notes for professor review.
# ======================================================================================

def assign_split(date_value, split_rules):
    """Assign train / validation / test based on locked date windows."""
    d = pd.to_datetime(date_value)
    for split_name, (start, end) in split_rules.items():
        if pd.to_datetime(start) <= d <= pd.to_datetime(end):
            return split_name
    return "outside_window"

def dataframe_to_records(df: pd.DataFrame, max_rows=None):
    """Convert dataframe to JSON-safe list of records."""
    tmp = df.copy()
    for col in tmp.columns:
        if pd.api.types.is_datetime64_any_dtype(tmp[col]):
            tmp[col] = tmp[col].dt.strftime("%Y-%m-%d")
    if max_rows is not None:
        tmp = tmp.head(max_rows)
    return tmp.replace({np.nan: None}).to_dict(orient="records")

def json_safe(obj):
    """Make nested objects JSON serializable."""
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if math.isnan(float(obj)):
            return None
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    return obj

# ------------------------------------------------------------------
# Load and validate input matrix
# ------------------------------------------------------------------

raw = pd.read_csv(INPUT_MATRIX_PATH)

if "date" not in raw.columns:
    raise ValueError("Input matrix must contain a 'date' column.")

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"]).copy()
raw = raw.sort_values("date").drop_duplicates(subset=["date"], keep="last").reset_index(drop=True)

required_base_columns = ["date", "gold_price"] + CORE_FACTORS
missing_required_columns = [c for c in required_base_columns if c not in raw.columns]
if missing_required_columns:
    raise ValueError(f"Missing required columns for feature engineering: {missing_required_columns}")

# If the file is raw instead of weekday-clean, remove weekends again as a safety guard.
raw_weekend_rows = int((raw["date"].dt.weekday >= 5).sum())
if raw_weekend_rows > 0:
    print(f"⚠️ Input contains {raw_weekend_rows:,} Saturday/Sunday rows. Removing them again as safety guard.")
    df = raw[raw["date"].dt.weekday < 5].copy()
    input_was_already_weekday_clean = False
else:
    df = raw.copy()
    input_was_already_weekday_clean = True

df = df.sort_values("date").reset_index(drop=True)

# ------------------------------------------------------------------
# Create leakage-safe engineered gold features
# ------------------------------------------------------------------
# Rule:
# - Lag features use prior rows directly.
# - Return features are shifted by 1 row so they represent prior known returns.
# - Moving averages and rolling volatility use gold_price.shift(1), so current target is not included.

feature_df = df.copy()

feature_df["gold_lag_1"] = feature_df["gold_price"].shift(1)
feature_df["gold_lag_5"] = feature_df["gold_price"].shift(5)
feature_df["gold_lag_20"] = feature_df["gold_price"].shift(20)

feature_df["gold_return_1"] = feature_df["gold_price"].pct_change(1).shift(1)
feature_df["gold_return_5"] = feature_df["gold_price"].pct_change(5).shift(1)

feature_df["gold_ma_5"] = feature_df["gold_price"].shift(1).rolling(window=5, min_periods=5).mean()
feature_df["gold_ma_20"] = feature_df["gold_price"].shift(1).rolling(window=20, min_periods=20).mean()
feature_df["gold_ma_60"] = feature_df["gold_price"].shift(1).rolling(window=60, min_periods=60).mean()

feature_df["gold_volatility_20"] = (
    feature_df["gold_price"]
    .pct_change(1)
    .shift(1)
    .rolling(window=20, min_periods=20)
    .std()
)

# ------------------------------------------------------------------
# Dataset A — Long univariate
# ------------------------------------------------------------------

long_mask = (
    (feature_df["date"] >= LONG_UNIVARIATE_WINDOW["start"]) &
    (feature_df["date"] <= LONG_UNIVARIATE_WINDOW["end"])
)

model_ready_univariate = feature_df.loc[long_mask, ["date", "gold_price"]].copy()
model_ready_univariate["split"] = model_ready_univariate["date"].apply(
    lambda d: assign_split(d, SPLIT_RULES["long_univariate"])
)
model_ready_univariate["dataset_id"] = "long_univariate"

# Keep readable column order.
model_ready_univariate = model_ready_univariate[["date", "dataset_id", "split", "gold_price"]]

# ------------------------------------------------------------------
# Dataset B — Core multivariate
# ------------------------------------------------------------------

multivariate_columns = ["date", "gold_price"] + XGBOOST_MAIN_FEATURES

multi_mask = (
    (feature_df["date"] >= CORE_MULTIVARIATE_WINDOW["start"]) &
    (feature_df["date"] <= CORE_MULTIVARIATE_WINDOW["end"])
)

model_ready_multivariate_before_drop = feature_df.loc[multi_mask, multivariate_columns].copy()
model_ready_multivariate_before_drop["split"] = model_ready_multivariate_before_drop["date"].apply(
    lambda d: assign_split(d, SPLIT_RULES["core_multivariate"])
)
model_ready_multivariate_before_drop["dataset_id"] = "core_multivariate"

required_multivariate_non_null = ["gold_price"] + XGBOOST_MAIN_FEATURES
multivariate_missing_before_drop = (
    model_ready_multivariate_before_drop[required_multivariate_non_null]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

model_ready_multivariate = model_ready_multivariate_before_drop.dropna(
    subset=required_multivariate_non_null
).copy()

# Keep readable column order.
model_ready_multivariate = model_ready_multivariate[
    ["date", "dataset_id", "split", "gold_price"] + XGBOOST_MAIN_FEATURES
].reset_index(drop=True)

# ------------------------------------------------------------------
# Split counts
# ------------------------------------------------------------------

univariate_split_counts = model_ready_univariate["split"].value_counts().to_dict()
multivariate_split_counts = model_ready_multivariate["split"].value_counts().to_dict()

# ------------------------------------------------------------------
# Feature dictionary
# ------------------------------------------------------------------

feature_dictionary = {
    "artifact_name": "feature_dictionary",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_variable": "gold_price",
    "leakage_policy": {
        "summary": "Gold-derived features are shifted so the current target value is not used to predict itself.",
        "gold_features": "lag/return/rolling features use only prior observed gold_price values.",
        "exogenous_factors": "Core macro/market factors are aligned by date inside the official cutoff window.",
        "cutoff_rule": f"No rows after {OFFICIAL_FORECAST_CUTOFF_DATE.date()} are used in model-ready datasets."
    },
    "engineered_gold_features": [
        {
            "feature": "gold_lag_1",
            "formula": "gold_price.shift(1)",
            "meaning": "Prior weekday gold price",
            "leakage_control": "Uses previous row only"
        },
        {
            "feature": "gold_lag_5",
            "formula": "gold_price.shift(5)",
            "meaning": "Gold price five weekday rows earlier",
            "leakage_control": "Uses previous rows only"
        },
        {
            "feature": "gold_lag_20",
            "formula": "gold_price.shift(20)",
            "meaning": "Gold price twenty weekday rows earlier",
            "leakage_control": "Uses previous rows only"
        },
        {
            "feature": "gold_return_1",
            "formula": "gold_price.pct_change(1).shift(1)",
            "meaning": "Prior 1-period gold return",
            "leakage_control": "Return is shifted by one row"
        },
        {
            "feature": "gold_return_5",
            "formula": "gold_price.pct_change(5).shift(1)",
            "meaning": "Prior 5-period gold return",
            "leakage_control": "Return is shifted by one row"
        },
        {
            "feature": "gold_ma_5",
            "formula": "gold_price.shift(1).rolling(5).mean()",
            "meaning": "Prior 5-row moving average",
            "leakage_control": "Rolling window begins from prior row"
        },
        {
            "feature": "gold_ma_20",
            "formula": "gold_price.shift(1).rolling(20).mean()",
            "meaning": "Prior 20-row moving average",
            "leakage_control": "Rolling window begins from prior row"
        },
        {
            "feature": "gold_ma_60",
            "formula": "gold_price.shift(1).rolling(60).mean()",
            "meaning": "Prior 60-row moving average",
            "leakage_control": "Rolling window begins from prior row"
        },
        {
            "feature": "gold_volatility_20",
            "formula": "gold_price.pct_change(1).shift(1).rolling(20).std()",
            "meaning": "Prior 20-row rolling volatility of gold returns",
            "leakage_control": "Return series is shifted before rolling"
        }
    ],
    "core_factor_features": [
        {
            "feature": factor,
            "source_type": "existing matrix factor",
            "main_multivariate_use": True
        }
        for factor in CORE_FACTORS
    ],
    "excluded_from_main_multivariate": [
        {
            "feature": "high_yield",
            "reason": "Starts too late around 2023-05-01; reserved for optional short-window sensitivity only."
        }
    ],
    "model_feature_sets": {
        "regression_recommended_first_pass": [
            "gold_lag_1",
            "gold_ma_20",
            "real_yield",
            "nominal_yield",
            "usd_index",
            "vix_index",
            "fin_stress",
            "gpr_index",
            "policy_unc",
            "oil_wti",
            "gld_tonnes",
            "unrate"
        ],
        "sarimax_recommended_exogenous": [
            "real_yield",
            "usd_index",
            "vix_index",
            "gpr_index",
            "policy_unc",
            "gld_tonnes",
            "oil_wti"
        ],
        "xgboost_main_candidate": XGBOOST_MAIN_FEATURES
    }
}

# ------------------------------------------------------------------
# Audit artifact
# ------------------------------------------------------------------

feature_engineering_audit = {
    "artifact_name": "feature_engineering_audit",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "03_feature_engineering_for_forecasting.ipynb",
    "input": {
        "input_matrix_path": str(INPUT_MATRIX_PATH),
        "input_rows": int(raw.shape[0]),
        "input_columns": int(raw.shape[1]),
        "input_date_min": raw["date"].min().strftime("%Y-%m-%d"),
        "input_date_max": raw["date"].max().strftime("%Y-%m-%d"),
        "input_was_already_weekday_clean": bool(input_was_already_weekday_clean),
        "weekend_rows_removed_in_this_notebook": int(raw_weekend_rows)
    },
    "cutoff": {
        "official_forecast_cutoff_date": OFFICIAL_FORECAST_CUTOFF_DATE.strftime("%Y-%m-%d"),
        "rows_after_cutoff_excluded_from_model_ready_outputs": int((df["date"] > OFFICIAL_FORECAST_CUTOFF_DATE).sum())
    },
    "features": {
        "engineered_gold_feature_count": len(ENGINEERED_GOLD_FEATURES),
        "core_factor_count": len(CORE_FACTORS),
        "main_xgboost_feature_count": len(XGBOOST_MAIN_FEATURES),
        "engineered_gold_features": ENGINEERED_GOLD_FEATURES,
        "core_factors": CORE_FACTORS,
        "excluded_main_features": ["high_yield"]
    },
    "datasets": {
        "model_ready_univariate": {
            "path": "data/aligned/model_ready_univariate.csv",
            "rows": int(model_ready_univariate.shape[0]),
            "columns": int(model_ready_univariate.shape[1]),
            "date_min": model_ready_univariate["date"].min().strftime("%Y-%m-%d"),
            "date_max": model_ready_univariate["date"].max().strftime("%Y-%m-%d"),
            "split_counts": {str(k): int(v) for k, v in univariate_split_counts.items()},
            "target": "gold_price",
            "used_for": ["Naive", "Moving Average", "Exponential Smoothing", "ARIMA", "Prophet optional"]
        },
        "model_ready_multivariate": {
            "path": "data/aligned/model_ready_multivariate.csv",
            "rows_before_missing_drop": int(model_ready_multivariate_before_drop.shape[0]),
            "rows_after_missing_drop": int(model_ready_multivariate.shape[0]),
            "rows_dropped_for_missing_required_values": int(model_ready_multivariate_before_drop.shape[0] - model_ready_multivariate.shape[0]),
            "columns": int(model_ready_multivariate.shape[1]),
            "date_min": model_ready_multivariate["date"].min().strftime("%Y-%m-%d"),
            "date_max": model_ready_multivariate["date"].max().strftime("%Y-%m-%d"),
            "split_counts": {str(k): int(v) for k, v in multivariate_split_counts.items()},
            "target": "gold_price",
            "used_for": ["Regression", "SARIMAX", "XGBoost main candidate"],
            "high_yield_excluded": True
        }
    },
    "missing_before_multivariate_drop": {
        str(k): int(v) for k, v in multivariate_missing_before_drop.items() if int(v) > 0
    },
    "professor_safe_notes": [
        "This notebook does not fit a model.",
        "The long univariate dataset contains gold_price only.",
        "The core multivariate dataset excludes high_yield from the main model path.",
        "Gold-derived rolling and return features are shifted to avoid using the current target value.",
        "No rows after the official forecast cutoff are included in model-ready outputs."
    ]
}

# ------------------------------------------------------------------
# Preview artifact
# ------------------------------------------------------------------

model_ready_dataset_preview = {
    "artifact_name": "model_ready_dataset_preview",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_ready_univariate": {
        "columns": list(model_ready_univariate.columns),
        "row_count": int(model_ready_univariate.shape[0]),
        "first_10_rows": dataframe_to_records(model_ready_univariate.head(10)),
        "last_10_rows": dataframe_to_records(model_ready_univariate.tail(10))
    },
    "model_ready_multivariate": {
        "columns": list(model_ready_multivariate.columns),
        "row_count": int(model_ready_multivariate.shape[0]),
        "first_10_rows": dataframe_to_records(model_ready_multivariate.head(10)),
        "last_10_rows": dataframe_to_records(model_ready_multivariate.tail(10))
    },
    "split_counts": {
        "long_univariate": {str(k): int(v) for k, v in univariate_split_counts.items()},
        "core_multivariate": {str(k): int(v) for k, v in multivariate_split_counts.items()}
    }
}

print("✅ Feature engineering complete")
print("Long univariate shape:", model_ready_univariate.shape)
print("Core multivariate shape:", model_ready_multivariate.shape)
print("Long split counts:", univariate_split_counts)
print("Multivariate split counts:", multivariate_split_counts)
print("Rows dropped from multivariate for missing values:", feature_engineering_audit["datasets"]["model_ready_multivariate"]["rows_dropped_for_missing_required_values"])

display(model_ready_univariate.head())
display(model_ready_multivariate.head())


In [ ]:
# ======================================================================================
# CELL 5 — ARTIFACT EXPORT
# ======================================================================================
# Purpose:
# - Save model-ready CSV datasets.
# - Save JSON artifacts used by later notebooks and frontend pages.
# ======================================================================================

# Make sure output folders exist.
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)
DATA_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Export CSVs.
univariate_csv_path = ALIGNED_DIR / "model_ready_univariate.csv"
multivariate_csv_path = ALIGNED_DIR / "model_ready_multivariate.csv"

model_ready_univariate_export = model_ready_univariate.copy()
model_ready_multivariate_export = model_ready_multivariate.copy()

model_ready_univariate_export["date"] = model_ready_univariate_export["date"].dt.strftime("%Y-%m-%d")
model_ready_multivariate_export["date"] = model_ready_multivariate_export["date"].dt.strftime("%Y-%m-%d")

model_ready_univariate_export.to_csv(univariate_csv_path, index=False)
model_ready_multivariate_export.to_csv(multivariate_csv_path, index=False)

# Export JSON artifacts.
feature_dictionary_path = DATA_ARTIFACTS_DIR / "feature_dictionary.json"
feature_engineering_audit_path = DATA_ARTIFACTS_DIR / "feature_engineering_audit.json"
model_ready_dataset_preview_path = DATA_ARTIFACTS_DIR / "model_ready_dataset_preview.json"

with open(feature_dictionary_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(feature_dictionary), f, indent=2)

with open(feature_engineering_audit_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(feature_engineering_audit), f, indent=2)

with open(model_ready_dataset_preview_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(model_ready_dataset_preview), f, indent=2)

print("✅ Exported:")
print("-", univariate_csv_path)
print("-", multivariate_csv_path)
print("-", feature_dictionary_path)
print("-", feature_engineering_audit_path)
print("-", model_ready_dataset_preview_path)


In [ ]:
# ======================================================================================
# CELL 6 — GITHUB PUSH
# ======================================================================================
# Purpose:
# - Commit and push Notebook 03 outputs to GitHub.
# ======================================================================================

files_to_add = [
    "data/aligned/model_ready_univariate.csv",
    "data/aligned/model_ready_multivariate.csv",
    "artifacts/data/feature_dictionary.json",
    "artifacts/data/feature_engineering_audit.json",
    "artifacts/data/model_ready_dataset_preview.json",
]

print("📌 Git status before add:")
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

run_cmd(["git", "add"] + files_to_add, cwd=str(PROJECT_DIR))

commit_message = "Add model-ready feature engineering datasets and artifacts"
commit = run_cmd(["git", "commit", "-m", commit_message], cwd=str(PROJECT_DIR), allow_fail=True)

if commit.returncode != 0:
    print("ℹ️ No new changes to commit, or commit already exists.")

if GITHUB_TOKEN:
    run_cmd(["git", "push", "origin", BRANCH], cwd=str(PROJECT_DIR), allow_fail=False)
    print("✅ Pushed feature engineering artifacts to GitHub.")
else:
    print("⚠️ GITHUB_TOKEN missing. Skipped push. Files were still written locally inside the cloned repo.")

print("✅ Notebook 03 complete.")
